# Exploring LLMs with the Multi-Agent Coding Assistant

This notebook demonstrates how the project uses OpenAI models across its agent pipeline.

**Prerequisites:**
- `pip install -r requirements.txt openai jupyter`
- Set `OPENAI_API_KEY` in `.env`
- Optional: backend running at `http://localhost:8000`

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from backend.config import settings

print(f"Model: {settings.DEFAULT_MODEL}")
print(f"API key set: {bool(settings.OPENAI_API_KEY)}")

## 1. Direct LLM call (BaseAgent)

All agents share `BaseAgent` which wraps the OpenAI chat API.

In [ ]:
from backend.agents.base import BaseAgent

agent = BaseAgent("You are a helpful Python tutor. Keep answers brief.")
result = agent._chat("Explain list comprehensions in one sentence.")

print(result["content"])
print(f"\nTokens used: {result['tokens_used']}")

## 2. Individual agents

Each agent has a specialized system prompt.

In [ ]:
from backend.agents.planner import planner
from backend.agents.coder import coder

task = "Write a function to check if a number is even"

plan_result = planner.plan(task, language="python")
print("=== Planner ===")
print(plan_result["plan"][:500])
print(f"Tokens: {plan_result['tokens_used']}")

if plan_result["status"] == "success":
    code_result = coder.code(task, plan_result["plan"], language="python")
    print("\n=== Coder ===")
    print(code_result["code"][:500])
    print(f"Tokens: {code_result['tokens_used']}")

## 3. Full pipeline via API

Run the backend (`python scripts/run_backend.py`) then execute the cell below.

In [ ]:
import requests

BASE_URL = "http://localhost:8000"

try:
    resp = requests.post(f"{BASE_URL}/api/solve", json={
        "task": "Write a Python function to count vowels in a string",
        "language": "python",
        "use_knowledge": False,
    }, timeout=120)
    resp.raise_for_status()
    data = resp.json()

    print(f"Confidence: {data['confidence_score']}")
    print(f"Total tokens: {data['total_tokens_used']}")
    print(f"Cost estimate: ${data['estimated_cost_usd']:.4f}")
    print("\nSolution:")
    print(data["solution"])
except requests.ConnectionError:
    print("Backend not running. Start with: python scripts/run_backend.py")

## 4. Token usage comparison

Compare token consumption across a simple vs complex task.

In [ ]:
tasks = [
    "Write a hello world function",
    "Implement a binary search tree with insert, delete, and in-order traversal",
]

for task in tasks:
    result = planner.plan(task)
    print(f"Task: {task[:50]}...")
    print(f"  Planner tokens: {result.get('tokens_used', 'N/A')}")
    print()